Library Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from typing import Tuple, Dict, List
import pickle
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple, Optional, Dict
from tqdm import tqdm

UCI Adult Dataset Preprocessing

In [2]:
class AdultDataProcessor:
    """Preprocessor for Adult Income dataset with fairness tracking"""

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = []
        self.sensitive_indices = {}

    def load_data(self, filepath: str = None) -> pd.DataFrame:
        """Load Adult dataset from file or download"""

        # Column names for Adult dataset
        column_names = [
            'age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
            'income'
        ]

        if filepath and os.path.exists(filepath):
            df = pd.read_csv(filepath, names=column_names, skipinitialspace=True)
        else:
            # Download from UCI repository
            url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
            df = pd.read_csv(url, names=column_names, skipinitialspace=True, na_values='?')

        return df

    def preprocess(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, Dict]:
        """
        Preprocess Adult dataset with sensitive attribute tracking

        Returns:
            X: Feature matrix
            y: Labels
            metadata: Dictionary containing demographic information
        """

        # Remove missing values
        df = df.dropna()

        # Store original sensitive attributes
        sensitive_attrs = {
            'race': df['race'].copy(),
            'sex': df['sex'].copy(),
            'age': df['age'].copy()
        }

        # Create age groups
        sensitive_attrs['age_group'] = pd.cut(
            df['age'],
            bins=[0, 25, 40, 60, 100],
            labels=['18-25', '26-40', '41-60', '60+']
        )

        # Binary label: >50K = 1, <=50K = 0
        y = (df['income'].str.strip() == '>50K').astype(int).values

        # Separate features
        feature_cols = [col for col in df.columns if col not in ['income']]

        # Encode categorical variables
        df_encoded = df[feature_cols].copy()
        categorical_cols = df_encoded.select_dtypes(include=['object']).columns

        for col in categorical_cols:
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
            self.label_encoders[col] = le

        # Store feature names
        self.feature_names = df_encoded.columns.tolist()

        # Convert to numpy array
        X = df_encoded.values.astype(np.float32)

        # Normalize features
        X = self.scaler.fit_transform(X)

        # Store sensitive attribute indices
        for attr in ['race', 'sex', 'age']:
            if attr in self.feature_names:
                self.sensitive_indices[attr] = self.feature_names.index(attr)

        # Create metadata
        metadata = {
            'sensitive_attrs': sensitive_attrs,
            'feature_names': self.feature_names,
            'n_features': X.shape[1],
            'n_samples': X.shape[0],
            'class_distribution': {
                '<=50K': np.sum(y == 0),
                '>50K': np.sum(y == 1)
            }
        }

        return X, y, metadata

    def create_train_test_member_splits(
        self,
        X: np.ndarray,
        y: np.ndarray,
        metadata: Dict,
        train_ratio: float = 0.6,
        test_ratio: float = 0.2,
        member_ratio: float = 0.2
    ) -> Dict:
        """
        Create train/test splits and member/non-member sets
        Preserves demographic distribution
        """

        n_samples = len(X)

        # Calculate split sizes
        train_size = int(n_samples * train_ratio)
        test_size = int(n_samples * test_ratio)
        member_size = int(n_samples * member_ratio)

        # First split: train + member vs test
        X_trainmem, X_test, y_trainmem, y_test, idx_trainmem, idx_test = train_test_split(
            X, y, np.arange(n_samples),
            test_size=test_size,
            random_state=self.random_seed,
            stratify=y
        )

        # Second split: train vs member (for attack evaluation)
        # Calculate member ratio relative to the trainmem set
        member_ratio_relative = member_ratio / (train_ratio + member_ratio)
        X_train, X_member, y_train, y_member, idx_train, idx_member = train_test_split(
            X_trainmem, y_trainmem, idx_trainmem,
            test_size=member_ratio_relative,
            random_state=self.random_seed,
            stratify=y_trainmem
        )

        # Extract demographic info for each split
        def get_demographics(indices):
            demo_dict = {}
            for attr in metadata['sensitive_attrs'].keys():
                attr_data = metadata['sensitive_attrs'][attr]
                # Handle both pandas Series and numpy arrays
                if hasattr(attr_data, 'iloc'):
                    demo_dict[attr] = attr_data.iloc[indices].values
                else:
                    demo_dict[attr] = attr_data[indices]
            return demo_dict

        return {
            'X_train': X_train,
            'y_train': y_train,
            'X_test': X_test,
            'y_test': y_test,
            'X_member': X_member,
            'y_member': y_member,
            'X_nonmember': X_test,  # Non-members are from test set
            'y_nonmember': y_test,
            'demographics_train': get_demographics(idx_train),
            'demographics_test': get_demographics(idx_test),
            'demographics_member': get_demographics(idx_member),
            'demographics_nonmember': get_demographics(idx_test),
            'metadata': metadata
        }

    def save(self, filepath: str):
        """Save processor state"""
        with open(filepath, 'wb') as f:
            pickle.dump({
                'scaler': self.scaler,
                'label_encoders': self.label_encoders,
                'feature_names': self.feature_names,
                'sensitive_indices': self.sensitive_indices
            }, f)

    def load(self, filepath: str):
        """Load processor state"""
        with open(filepath, 'rb') as f:
            state = pickle.load(f)
            self.scaler = state['scaler']
            self.label_encoders = state['label_encoders']
            self.feature_names = state['feature_names']
            self.sensitive_indices = state['sensitive_indices']


def calculate_data_sensitivity(X: np.ndarray) -> float:
    """
    Calculate L2 sensitivity of the dataset for DP noise calibration

    Args:
        X: Feature matrix

    Returns:
        sensitivity: Maximum L2 norm of any sample
    """
    norms = np.linalg.norm(X, axis=1)
    return np.max(norms)


def analyze_demographic_distribution(demographics: Dict, name: str = "Dataset"):
    """Print demographic distribution statistics"""

    print(f"\n{name} Demographic Distribution:")
    print("=" * 60)

    for attr, values in demographics.items():
        if attr == 'age':
            print(f"\n{attr.upper()}:")
            print(f"  Mean: {np.mean(values):.2f}")
            print(f"  Std: {np.std(values):.2f}")
            print(f"  Range: [{np.min(values)}, {np.max(values)}]")
        else:
            unique, counts = np.unique(values, return_counts=True)
            print(f"\n{attr.upper()}:")
            for val, count in zip(unique, counts):
                print(f"  {val}: {count} ({count/len(values)*100:.1f}%)")

PATE Noise Mechanism Training

In [3]:
class PATE:
    """
    Private Aggregation of Teacher Ensembles
    Based on Papernot et al. (ICLR 2017)
    """

    def __init__(
        self,
        input_dim: int,
        hidden_layers: List[int],
        num_teachers: int = 10,
        epsilon: float = 1.0,
        delta: float = 1e-5,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.num_teachers = num_teachers
        self.epsilon = epsilon
        self.delta = delta
        self.device = device

        self.teachers = []
        self.student = None
        self.privacy_budget_spent = 0.0

    def _partition_data(
        self,
        X: np.ndarray,
        y: np.ndarray
    ) -> List[Tuple[np.ndarray, np.ndarray]]:
        """Partition data for teacher models"""

        n_samples = len(X)
        partition_size = n_samples // self.num_teachers

        # Shuffle data
        indices = np.random.permutation(n_samples)
        X_shuffled = X[indices]
        y_shuffled = y[indices]

        partitions = []
        for i in range(self.num_teachers):
            start_idx = i * partition_size
            end_idx = start_idx + partition_size if i < self.num_teachers - 1 else n_samples

            X_partition = X_shuffled[start_idx:end_idx]
            y_partition = y_shuffled[start_idx:end_idx]
            partitions.append((X_partition, y_partition))

        return partitions

    def train_teachers(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        epochs: int = 30,
        learning_rate: float = 0.01,
        batch_size: int = 256,
        verbose: bool = True
    ):
        """Train ensemble of teacher models"""

        if verbose:
            print(f"Training {self.num_teachers} teacher models...")

        partitions = self._partition_data(X_train, y_train)

        for i, (X_part, y_part) in enumerate(partitions):
            # Create validation split for each teacher
            X_train_t, X_val_t, y_train_t, y_val_t = train_test_split(
                X_part, y_part, test_size=0.2, random_state=i
            )

            # Create and train teacher (use DP-compatible architecture)
            teacher = MLP(self.input_dim, self.hidden_layers, use_dp=False).to(self.device)
            trainer = StandardTrainer(
                teacher,
                learning_rate=learning_rate,
                batch_size=batch_size,
                device=self.device
            )

            trainer.train(
                X_train_t, y_train_t,
                X_val_t, y_val_t,
                epochs=epochs,
                verbose=False
            )

            self.teachers.append(teacher)

            if verbose:
                print(f"  Teacher {i+1}/{self.num_teachers} trained")

    def _noisy_aggregation(self, votes: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Aggregate teacher predictions with Laplacian noise

        Args:
            votes: Array of shape (n_samples, n_classes) with teacher votes

        Returns:
            predictions: Noisy aggregated predictions
            privacy_cost: Privacy budget spent
        """

        # Count votes for each class
        vote_counts = votes.sum(axis=1)

        # Add Laplacian noise for privacy
        # Sensitivity is 1 (changing one teacher's vote changes count by 1)
        noise_scale = 1.0 / self.epsilon
        noise = np.random.laplace(0, noise_scale, size=vote_counts.shape)
        noisy_counts = vote_counts + noise

        # Make predictions based on noisy counts
        predictions = (noisy_counts > self.num_teachers / 2).astype(int)

        # Calculate privacy cost (simplified - use moments accountant for accuracy)
        privacy_cost = self.epsilon * len(votes) / 10000  # Approximate

        return predictions, privacy_cost

    def generate_student_labels(
        self,
        X_public: np.ndarray,
        verbose: bool = True
    ) -> Tuple[np.ndarray, float]:
        """
        Generate labels for student training using noisy aggregation

        Args:
            X_public: Public data for student training

        Returns:
            labels: Private labels for student
            privacy_spent: Total privacy budget spent
        """

        if not self.teachers:
            raise ValueError("Teachers must be trained first!")

        if verbose:
            print("Generating private labels for student...")

        # Get predictions from all teachers
        teacher_preds = []
        for teacher in self.teachers:
            teacher.eval()
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X_public).to(self.device)
                preds = (teacher(X_tensor).squeeze() > 0.5).cpu().numpy()
                teacher_preds.append(preds)

        teacher_preds = np.array(teacher_preds).T  # Shape: (n_samples, n_teachers)

        # Aggregate with noise
        labels, privacy_cost = self._noisy_aggregation(teacher_preds)
        self.privacy_budget_spent += privacy_cost

        if verbose:
            print(f"  Privacy budget spent: {privacy_cost:.6f}")
            print(f"  Total privacy spent: {self.privacy_budget_spent:.6f}")

        return labels, privacy_cost

    def train_student(
        self,
        X_public: np.ndarray,
        epochs: int = 50,
        learning_rate: float = 0.01,
        batch_size: int = 256,
        verbose: bool = True
    ):
        """Train student model on privately labeled public data"""

        if verbose:
            print("\nTraining student model...")

        # Generate private labels
        y_private, _ = self.generate_student_labels(X_public, verbose=False)

        # Split for validation
        X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
            X_public, y_private, test_size=0.2, random_state=42
        )

        # Create and train student (use DP-compatible architecture)
        self.student = MLP(self.input_dim, self.hidden_layers, use_dp=False).to(self.device)
        trainer = StandardTrainer(
            self.student,
            learning_rate=learning_rate,
            batch_size=batch_size,
            device=self.device
        )

        history = trainer.train(
            X_train_s, y_train_s,
            X_val_s, y_val_s,
            epochs=epochs,
            verbose=verbose
        )

        return history

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict using student model"""
        if self.student is None:
            raise ValueError("Student must be trained first!")

        self.student.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.student(X_tensor).squeeze()
            return (outputs > 0.5).cpu().numpy()

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict probabilities using student model"""
        if self.student is None:
            raise ValueError("Student must be trained first!")

        self.student.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.student(X_tensor).squeeze()
            return outputs.cpu().numpy()

    def get_sample_losses(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        """Get per-sample losses from student"""
        if self.student is None:
            raise ValueError("Student must be trained first!")

        self.student.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            y_tensor = torch.FloatTensor(y).to(self.device)
            losses = self.student.get_loss(X_tensor, y_tensor)
            return losses.cpu().numpy()

    def get_privacy_spent(self) -> float:
        """Return total privacy budget spent"""
        return self.privacy_budget_spent


class PATETrainer:
    """Wrapper for PATE to match the interface of other trainers"""

    def __init__(
        self,
        input_dim: int,
        hidden_layers: List[int],
        num_teachers: int = 10,
        epsilon: float = 1.0,
        delta: float = 1e-5,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.pate = PATE(
            input_dim=input_dim,
            hidden_layers=hidden_layers,
            num_teachers=num_teachers,
            epsilon=epsilon,
            delta=delta,
            device=device
        )
        self.training_history = {'loss': [], 'accuracy': []}

    def train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        epochs: int = 50,
        teacher_epochs: int = 30,
        verbose: bool = True
    ) -> Dict:
        """Train PATE model"""

        # Train teachers on private data
        self.pate.train_teachers(
            X_train, y_train,
            epochs=teacher_epochs,
            verbose=verbose
        )

        # Use validation set as "public" data for student
        # In practice, you would use actual public data
        history = self.pate.train_student(
            X_val,
            epochs=epochs,
            verbose=verbose
        )

        self.training_history = history
        return history

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.pate.predict(X)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.pate.predict_proba(X)

    def get_sample_losses(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        return self.pate.get_sample_losses(X, y)

MLP Model Definition w/ Standard Training & DP-SGD Training

In [4]:
class MLP(nn.Module):
    """Multi-layer perceptron for binary classification"""


    def __init__(self, input_dim: int, hidden_layers: List[int], dropout: float = 0.3, use_dp: bool = False):
        super(MLP, self).__init__()

        # List to store sequence of layers
        # Defines input size of input layer (first layer)
        layers = []
        prev_dim = input_dim

        # Iterate through each hidden layer...
        # Create a fully connected layer by performing linear transformation with weights and biases
        # ReLU activation function allows for network to learn complex relationships.
        for hidden_dim in hidden_layers:

            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())

            # Use GroupNorm for DP compatibility, BatchNorm otherwise
            if use_dp:
                # GroupNorm: divide channels into groups (use 1 group = LayerNorm behavior for 1D)
                num_groups = min(32, hidden_dim)  # Ensure divisibility
                while hidden_dim % num_groups != 0:
                    num_groups -= 1
                layers.append(nn.GroupNorm(num_groups, hidden_dim))
            else:
                layers.append(nn.BatchNorm1d(hidden_dim))

            # Dropout of 30% to prevent overfitting (sets 30% of inputs to values of 0)
            layers.append(nn.Dropout(dropout))

            # Output dimention of current layer is assigned as the input dimention for next layer
            prev_dim = hidden_dim

        # Output layer - condenses output into a single number
        layers.append(nn.Linear(prev_dim, 1))

        # Converts to a single number between 0 and 1 (i.e. probability)
        layers.append(nn.Sigmoid())

        # Combines all layers into a single container
        self.network = nn.Sequential(*layers)

    # Defines how a given input flows through the network
    # self.network(x) enables the feeding of the input data (x) through each sequence of the layers
    # returns tensor of logits (i.e. raw prediction scores)
    def forward(self, x):
        return self.network(x)

    # Utility Function: Computes the prediction confidence for an input (x) using PyTorch model
    # To start torch.no_grad() temporarily sets requires_grad flags to False to avoid calculating gradients
    def get_confidence(self, x):
        """Get prediction confidence (probability)"""
        with torch.no_grad():
            return self.forward(x)

    # Computes the loss for each individual input sample in a given batch as part of the neural network training.
    # Pass input (x) through all network layers and return raw prediction scores
    # Compute the loss/error between prediction and true label
    # Leveraging the Binary Cross-Entropr Loss function (pred = predicted probability, y = true label)
    def get_loss(self, x, y):
        """Get per-sample loss"""
        pred = self.forward(x)
        loss = nn.BCELoss(reduction='none')(pred, y.unsqueeze(1).float())

        # Return the loss value for every sample (as a 1D tensor)
        return loss.squeeze()


class StandardTrainer:
    """Standard (non-private) model trainer"""

    def __init__(
        self,
        model: nn.Module,
        learning_rate: float = 0.01,
        batch_size: int = 256,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.model = model.to(device)
        self.device = device
        self.batch_size = batch_size
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.criterion = nn.BCELoss()
        self.training_history = {'loss': [], 'accuracy': []}

    # Conducts model training, returns dictionary
    def train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        epochs: int = 50,
        verbose: bool = True
    ) -> Dict:
        """Train the model"""

        # Check to ensure inputs and labels match
        assert len(X_train) == len(y_train), f"Training mismatch: X({len(X_train)}) != y({len(y_train)})"

        # Handle None validation data
        if X_val is None or y_val is None:
            # Use a small portion of training data for validation
            split_idx = int(0.8 * len(X_train))
            X_val = X_train[split_idx:]
            y_val = y_train[split_idx:]
            X_train = X_train[:split_idx]
            y_train = y_train[:split_idx]

        # This is where your current code is likely failing silently until the loss calculation
        if len(X_val) != len(y_val):
            print(f"Warning: Validation mismatch detected! X({len(X_val)}) vs y({len(y_val)})")
            # Force trim to the smaller size to prevent crash
            min_len = min(len(X_val), len(y_val))
            X_val = X_val[:min_len]
            y_val = y_val[:min_len]

        # Convert to tensors
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train),
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        X_val_tensor = torch.FloatTensor(X_val).to(self.device)
        y_val_tensor = torch.FloatTensor(y_val).to(self.device)

        for epoch in range(epochs):
            self.model.train()
            epoch_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(batch_X).squeeze(1) # squeeze(1) only remove the feature dimension
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()

                epoch_loss += loss.item()

            # Validation
            self.model.eval()
            with torch.no_grad():
                val_outputs = self.model(X_val_tensor).squeeze()
                val_loss = self.criterion(val_outputs, y_val_tensor).item()
                val_preds = (val_outputs > 0.5).float()
                val_acc = (val_preds == y_val_tensor).float().mean().item()

            self.training_history['loss'].append(epoch_loss / len(train_loader))
            self.training_history['accuracy'].append(val_acc)

            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - "
                      f"Loss: {epoch_loss/len(train_loader):.4f} - "
                      f"Val Acc: {val_acc:.4f}")

        return self.model, self.training_history

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict labels"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor).squeeze()
            return (outputs > 0.5).cpu().numpy()

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict probabilities"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor).squeeze()
            return outputs.cpu().numpy()

    def get_sample_losses(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        """Get per-sample losses for attack"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            y_tensor = torch.FloatTensor(y).to(self.device)
            losses = self.model.get_loss(X_tensor, y_tensor)
            return losses.cpu().numpy()


class DPSGDTrainer:
    """Differential Privacy SGD trainer using Opacus"""

    def __init__(
        self,
        model: nn.Module,
        epsilon: float,
        delta: float = 1e-5,
        max_grad_norm: float = 1.0,
        learning_rate: float = 0.01,
        batch_size: int = 256,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.model = model.to(device)
        self.device = device
        self.batch_size = batch_size
        self.epsilon = epsilon
        self.delta = delta
        self.max_grad_norm = max_grad_norm
        self.training_history = {'loss': [], 'accuracy': [], 'epsilon': []}

        try:
            from opacus import PrivacyEngine
            self.privacy_engine = PrivacyEngine()
            self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            self.use_opacus = True
        except ImportError:
            print("Warning: Opacus not available. Using manual DP-SGD implementation.")
            self.use_opacus = False
            self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            self.noise_multiplier = self._calculate_noise_multiplier()

        self.criterion = nn.BCELoss()

    def _calculate_noise_multiplier(self) -> float:
        """Calculate noise multiplier for target epsilon"""
        # Simplified calculation - use proper privacy accounting in production
        return np.sqrt(2 * np.log(1.25 / self.delta)) / self.epsilon

    def _clip_gradients(self):
        """Manually clip gradients"""
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5

        clip_coef = self.max_grad_norm / (total_norm + 1e-6)
        if clip_coef < 1:
            for p in self.model.parameters():
                if p.grad is not None:
                    p.grad.data.mul_(clip_coef)

    def _add_noise(self):
        """Add Gaussian noise to gradients"""
        for p in self.model.parameters():
            if p.grad is not None:
                noise = torch.randn_like(p.grad) * self.noise_multiplier * self.max_grad_norm
                p.grad.data.add_(noise)

    def train(self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        epochs: int = 50,
        verbose: bool = True
    ) -> Dict:
        """Train with differential privacy"""

        train_dataset = TensorDataset(
            torch.FloatTensor(X_train),
            torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        if self.use_opacus:
            # Attach privacy engine
            self.model, self.optimizer, train_loader = self.privacy_engine.make_private(
                module=self.model,
                optimizer=self.optimizer,
                data_loader=train_loader,
                noise_multiplier=self._calculate_noise_multiplier(),
                max_grad_norm=self.max_grad_norm,
            )

        X_val_tensor = torch.FloatTensor(X_val).to(self.device)
        y_val_tensor = torch.FloatTensor(y_val).to(self.device)

        for epoch in range(epochs):
            self.model.train()
            epoch_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(batch_X).squeeze()
                loss = self.criterion(outputs, batch_y)
                loss.backward()

                if not self.use_opacus:
                    self._clip_gradients()
                    self._add_noise()

                self.optimizer.step()
                epoch_loss += loss.item()

            # Validation
            self.model.eval()
            with torch.no_grad():
                val_outputs = self.model(X_val_tensor).squeeze()
                val_preds = (val_outputs > 0.5).float()
                val_acc = (val_preds == y_val_tensor).float().mean().item()

            # Track privacy budget
            if self.use_opacus:
                epsilon_spent = self.privacy_engine.get_epsilon(self.delta)
            else:
                # Approximate epsilon accounting
                epsilon_spent = self.epsilon * (epoch + 1) / epochs

            self.training_history['loss'].append(epoch_loss / len(train_loader))
            self.training_history['accuracy'].append(val_acc)
            self.training_history['epsilon'].append(epsilon_spent)

            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - "
                      f"Loss: {epoch_loss/len(train_loader):.4f} - "
                      f"Val Acc: {val_acc:.4f} - "
                      f"ε: {epsilon_spent:.4f}")

        return self.model, self.training_history

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict labels"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor).squeeze()
            return (outputs > 0.5).cpu().numpy()

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict probabilities"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor).squeeze()
            return outputs.cpu().numpy()

    def get_sample_losses(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        """Get per-sample losses"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            y_tensor = torch.FloatTensor(y).to(self.device)

            # Access underlying model if wrapped by Opacus
            if hasattr(self.model, '_module'):
                # Opacus wrapped model
                base_model = self.model._module
            else:
                base_model = self.model

            # Calculate loss directly since get_loss might not be accessible
            outputs = base_model(X_tensor).squeeze()
            loss = nn.BCELoss(reduction='none')(outputs, y_tensor)
            return loss.cpu().numpy()


def create_model(input_dim: int, hidden_layers: List[int], dropout: float = 0.3, use_dp: bool = False) -> MLP:
    """Factory function to create a new model"""
    return MLP(input_dim, hidden_layers, dropout, use_dp)

Label-Only Based MIA

In [5]:
class LabelOnlyMIA:

    # Default constructor
    # Noise scale set to 1.0 based on basic knowledge of dataset
    # No.of augmentations (amt. of time a new perturbed input is queried).
    def __init__(self, model: nn.Module, n_augmentations: int = 100, noise_scale: float = 1.0):
        self.model = model
        self.model.eval()
        self.n_augmentations = n_augmentations
        self.noise_scale = noise_scale
        self.device = next(model.parameters()).device  # Store current device the model is on (CPU or GPU) to apply to downstream resources

    # Method queries model for its prediction consistency.
    # Returns consistency score (avg) & pertubed predictions
    def compute_label_prediction_consistency(self, x_input):

        # Sanity check to ensure input (X) has been convert to tensor
        if not isinstance(x_input, torch.Tensor):
          x_input = torch.FloatTensor(x_input)

        # Move tensor to the same device as the model to avoid runtime errors
        x_input = x_input.to(self.device)

        # Ensure input has batch dimension
        if x_input.dim() == 1:
            x_input = x_input.unsqueeze(0)

        # Perform forward pass through neural network to get original predictions w/o obtaining gradient calculations
        with torch.no_grad():
          orig_logits = self.model(x_input)
          orig_pred = torch.argmax(orig_logits, dim=1).item()

        # List for pertubed predictions returned
        predictions = []

        # Initialize counter
        pred_consistency_cnt = 0

        # Loop base on no.of augmentations...
        # Create individual pertubations using gaussian noise
        # Add noise to input (x)
        # Perform forward pass through neural network to get perturbed predictions w/o obtaining gradient calculations
        for aug in range(self.n_augmentations):
          noise = torch.normal(mean=0, std=self.noise_scale, size=x_input.shape).to(self.device)
          x_perturbed = x_input + noise

          with torch.no_grad():
            perturbed_logits = self.model(x_perturbed)
            perturbed_pred = torch.argmax(perturbed_logits, dim=1).item()

          predictions.append(perturbed_pred)

          # Check if new prediction equals the original prediction
          # If true, increment consistency counter (signifying that its a member)
          if perturbed_pred == orig_pred:
            pred_consistency_cnt +=1

        # Returns consistency average & pertubed predictions
        consistency_avg = pred_consistency_cnt / self.n_augmentations

        return consistency_avg, predictions


    # Performs the attack
    def perform_label_attack_members(self, X_train, n_samples):

      members_consistency = []
      total_train_samples = min(n_samples, len(X_train))


      for i in range(total_train_samples):
        consistency, preds = self.compute_label_prediction_consistency(X_train[i])
        members_consistency.append(consistency)

        # Print progress every 1000 samples
        if (i + 1) % 1000 == 0 or (i + 1) == total_train_samples:

            # Calculate percentage completion
            percent_complete = ((i + 1) / total_train_samples) * 100

            # Use carriage return (\r) to print on the same line (makes it look like a dynamic bar)
            print(f"\rProcessed: {i + 1}/{total_train_samples} samples from Members ({percent_complete:.1f}%)")

      return members_consistency


    def perform_label_attack_nonmembers(self, X_test, n_samples):

      non_members_consistency = []
      total_test_samples = min(n_samples, len(X_test))

      for i in range(total_test_samples):
        consistency, preds = self.compute_label_prediction_consistency(X_test[i])
        non_members_consistency.append(consistency)

        # Print progress every 1000 samples
        if (i + 1) % 1000 == 0 or (i + 1) == total_test_samples:

            # Calculate percentage completion
            percent_complete = ((i + 1) / total_test_samples) * 100

            # Use carriage return (\r) to print on the same line (makes it look like a dynamic bar)
            print(f"\rProcessed: {i + 1}/{total_test_samples} samples from Non-members ({percent_complete:.1f}%)")

      return non_members_consistency

Prediction-Loss & Confidence-Based MIA

In [14]:
# Assumption is that the attack won't know that the published data had DP-SGD training but only Standard MLP training, thus why StandardTrainer is inherited
class ThresholdBasedMIA():

  # Default constructor
  # Define target model, no.of shadow models to be created, training & test sets
  def __init__(self,
               target_trainer,
               X_train: np.ndarray,
               y_train: np.ndarray,
               X_test: np.ndarray,
               y_test: np.ndarray,
               n_shadows: int = 3,
               input_dim: int = None,
               hidden_layers: list = None):

    # Define train target model features / labels
    self.target_trainer = target_trainer
    self.target_model = target_trainer.model
    self.n_shadows = n_shadows
    self.X_train = X_train
    self.y_train = y_train
    self.X_test = X_test
    self.y_test = y_test

    # Store model architecture params
    self.input_dim = input_dim if input_dim else X_train.shape[1]
    self.hidden_layers = hidden_layers if hidden_layers else [32, 16]

    # Declare model list obj
    self.shadow_models = []

  # Compute the prediction loss metric for each sample (row)
  # Input params: trainer, features, labels
  def compute_metric_loss(self, trainer, X, y):

    # Epilson value for numerical stability
    epsilon = 1e-10

    # Compute the probability for each sample (row) with respect to the feature classification
    probs = trainer.predict_proba(X)

    if probs.ndim == 1:
        # probs represents probability of class 1
        # Convert to 2D format: [prob_class_0, prob_class_1]
        probs = np.column_stack([1 - probs, probs])

    # Get probability of the true class for each sample
    true_feature_probs = probs[np.arange(len(y)), y]

    # Compute Cross-Entropy Loss for each sample
    # A lower loss value means a better, more confident prediction.
    loss_vals = -np.log(true_feature_probs + epsilon)

    # Return loss
    return loss_vals


  # Compute the prediction loss metric for each sample (row)
  # Input params: trainer, features, labels
  def compute_prediction_confidence(self, trainer, X, y):

    # Epilson value for numerical stability
    epsilon = 1e-10

    # Compute the probability for each sample (row) with respect to the feature classification
    probs = trainer.predict_proba(X)

    if probs.ndim == 1:
        # probs represents probability of class 1
        # Convert to 2D format: [prob_class_0, prob_class_1]
        probs = np.column_stack([1 - probs, probs])

    # Get total samples
    samples = len(X)

    # Get maximum probability of predicted class
    max_confidence = np.max(probs, axis=1)

    # Get probability of the true class for each sample
    true_feature_confidence = probs[np.arange(samples), y]

    sorted_probs = np.sort(probs, axis=1)
    confidence_margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    # Compute Cross-Entropy of predictions for each sample
    entropy = -np.sum(probs * np.log(probs + epsilon), axis=1)

    # Combine all features
    features = np.column_stack([
        max_confidence,
        true_feature_confidence,
        confidence_margin,
        entropy
    ])

    return features

  # Train shadow model
  # Input params: D-shadow
  # Rule of thumb: Split D shadow (same sample of data used to train_target_model) into train_set (members) & test_set (nonmembers)
  def train_shadow_model(self, X_member, y_member, X_nonmember, y_nonmember):

    # Create list to store shadow model data
    d_train_data = []
    d_test_data = []

    # Train shadow model for the total specified in class obj instantiation
    # If n_shadows = 5, train 5 diferent models
    for i in range(self.n_shadows):

      # Create new model with same architecture
      new_model = MLP(input_dim=self.input_dim, hidden_layers=self.hidden_layers)
      shadow_trainer = StandardTrainer(model=new_model)

      # Conduct MLP training using StandardTrainer class
      model_trained, model_history = shadow_trainer.train(
          X_member, y_member.ravel(),
          X_nonmember, y_nonmember.ravel()
      )

      # Add save shadow model this instance of class list for easy accessibility within other functions
      self.shadow_models.append(shadow_trainer)

      # Store the train/test splits with their associated labels
      d_train_data.append((X_member, y_member, shadow_trainer))
      d_test_data.append((X_nonmember, y_nonmember, shadow_trainer))

      # Display shadow model training process
      if (i + 1) % self.n_shadows == 0:
        print(f"Trained {i + 1}/{self.n_shadows} shadow models")

    # Return shadow models trained via list
    return d_train_data, d_test_data

  # Prepare dataset for which the attack model was target
  # Dataset includes prediction loss values
  def prep_attack_dataset(self, d_train_data, d_test_data, threshold_type):

    # Single lists for storing prediction loss value
    X_attack = []
    y_attack = []

    # For the values in the d_train_data (member data ~ label = 1)
    # Calculate the metric loss or prediction confidence
    # Values will be in a 1D vector (e.g. [0.3, 0.4] etc.)
    for X_train, y_train, shadow_trainer in d_train_data:

      if threshold_type == 'loss':
        vals = self.compute_metric_loss(shadow_trainer, X_train, y_train)
      else:
        vals = self.compute_prediction_confidence(shadow_trainer, X_train, y_train)

      # Flatten to ensure loss_vals is in 1D array format
      computed_vals = np.array(vals).flatten()

      # Add loss values to list
      X_attack.extend(computed_vals)

      # Because the assumption is that the training set will be all members
      # We create a list of ones based on the no.of samples being computed for prediction loss
      y_attack.extend([1] * len(computed_vals))


    # For the values in the d_test_data (nonmember data ~ label = 0)
    # Calculate the metric loss or prediction confidence
    # Values will be in a 1D vector (e.g. [0.3, 0.4] etc.)
    for X_train, y_train, shadow_trainer in d_test_data:

      if threshold_type == 'loss':
        vals = self.compute_metric_loss(shadow_trainer, X_train, y_train)

      elif threshold_type == 'confidence':
        vals = self.compute_prediction_confidence(shadow_trainer, X_train, y_train)

      # Flatten to ensure loss_vals is in 1D array format
      computed_vals = np.array(vals).flatten()

      # Add loss values to list
      X_attack.extend(computed_vals)

      # Because the assumption is that the training set will be all members
      # We create a list of ones based on the no.of samples being computed for prediction loss
      y_attack.extend([0] * len(computed_vals))

    X_attack = np.array(X_attack, dtype=np.float64).reshape(-1, 1)
    y_attack = np.array(y_attack, dtype=np.int32)

    # Return loss predictions & labels
    return X_attack, y_attack


  # Train the attack model
  # Input params: X_attack, y_attack
  def train_attack_model(self, X_attack, y_attack):

    # Create a simple MLP for attack model with correct input dimension
    # X_attack has shape (n_samples, 1) for loss-based attacks
    # or (n_samples, 4) for confidence-based attacks
    attack_input_dim = X_attack.shape[1]

    # Create small attack model
    attack_model = MLP(input_dim=attack_input_dim, hidden_layers=[32, 16])
    attack_trainer = StandardTrainer(model=attack_model)

    # Split for validation
    split_idx = int(0.8 * len(X_attack))
    X_train_attack = X_attack[:split_idx]
    y_train_attack = y_attack[:split_idx]
    X_val_attack = X_attack[split_idx:]
    y_val_attack = y_attack[split_idx:]

    # Train the attack model
    self.attack_model, model_history = attack_trainer.train(
        X_train_attack, y_train_attack.ravel(),
        X_val_attack, y_val_attack.ravel()
    )

    # Store the trainer so we can use predict_proba later
    self.attack_trainer = attack_trainer

    print(f"Attack model accuracy: {model_history['accuracy'][-1]:.2%}")


  # Perform Membership Inference attack on target model
  # Compute losses for target model's training data (actual members) & (actual non-members)
  def execute_attack(self, threshold_type):

    if threshold_type == 'loss':

      train_result = self.compute_metric_loss(self.target_model, self.X_train, self.y_train)
      test_result = self.compute_metric_loss(self.target_model, self.X_test, self.y_test)

    elif threshold_type == 'confidence':

      train_result = self.compute_prediction_confidence(self.target_model, self.X_train, self.y_train)
      test_result = self.compute_prediction_confidence(self.target_model, self.X_test, self.y_test)


    # Reshape to match expected input format
    train_result_reshaped = train_result.reshape(-1, 1)
    test_result_reshaped = test_result.reshape(-1, 1)

    # Use the trainer's predict method
    train_predictions = self.attack_trainer.predict(train_result_reshaped)
    test_predictions = self.attack_trainer.predict(test_result_reshaped)

    # Calculate attack accuracy
    train_acc = accuracy_score([1] * len(train_predictions), train_predictions)
    test_acc = accuracy_score([0] * len(test_predictions), test_predictions)
    overall_acc = (train_acc * len(train_predictions) + test_acc * len(test_predictions)) / \
                  (len(train_predictions) + len(test_predictions))

    return {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overall_accuracy': overall_acc,
        'train_result': train_result,
        'test_result': test_result,
        'train_predictions': train_predictions,
        'test_predictions': test_predictions
    }

Execute Pipeline Logic

In [7]:
if __name__ == "__main__":

  # Perform data processing on Adult Dataset
  processor = AdultDataProcessor(random_seed=42)

  print("Loading Adult dataset...")
  df = processor.load_data()
  print(f"Loaded {len(df)} records")

  print("\nPreprocessing...")
  X, y, metadata = processor.preprocess(df)
  print(f"Features: {X.shape[1]}")
  print(f"Samples: {X.shape[0]}")
  print(f"Class distribution: {metadata['class_distribution']}")

  # Store features & samples
  n_features = X.shape[1]
  n_samples = X.shape[0]

  print("\nCreating splits...")
  splits = processor.create_train_test_member_splits(X, y, metadata)

  print(f"\nTrain set: {len(splits['X_train'])} samples")
  print(f"Test set: {len(splits['X_test'])} samples")
  print(f"Member set: {len(splits['X_member'])} samples")
  print(f"Non-member set: {len(splits['X_nonmember'])} samples")

  # Analyze demographics
  analyze_demographic_distribution(splits['demographics_train'], "Training Set")
  analyze_demographic_distribution(splits['demographics_test'], "Test Set")

  # Calculate sensitivity
  sensitivity = calculate_data_sensitivity(X)
  print(f"\nData L2 sensitivity: {sensitivity:.4f}")

  # Store 4-way sub datasets
  # Separate training & test set for target model
  X_train = splits['X_train']
  y_train = splits['y_train']
  X_test = splits['X_test']
  y_test = splits['y_test']

  # Separate training & test set for shadow models
  X_member = splits['X_member']
  y_member = splits['y_member']
  X_nonmember = splits['X_member']
  y_nonmember = splits['y_test']

Loading Adult dataset...
Loaded 32561 records

Preprocessing...
Features: 14
Samples: 30162
Class distribution: {'<=50K': np.int64(22654), '>50K': np.int64(7508)}

Creating splits...

Train set: 18097 samples
Test set: 6032 samples
Member set: 6033 samples
Non-member set: 6032 samples

Training Set Demographic Distribution:

RACE:
  Amer-Indian-Eskimo: 178 (1.0%)
  Asian-Pac-Islander: 534 (3.0%)
  Black: 1701 (9.4%)
  Other: 140 (0.8%)
  White: 15544 (85.9%)

SEX:
  Female: 5841 (32.3%)
  Male: 12256 (67.7%)

AGE:
  Mean: 38.46
  Std: 13.04
  Range: [17, 90]

AGE_GROUP:
  18-25: 3371 (18.6%)
  26-40: 7221 (39.9%)
  41-60: 6490 (35.9%)
  60+: 1015 (5.6%)

Test Set Demographic Distribution:

RACE:
  Amer-Indian-Eskimo: 64 (1.1%)
  Asian-Pac-Islander: 186 (3.1%)
  Black: 558 (9.3%)
  Other: 38 (0.6%)
  White: 5186 (86.0%)

SEX:
  Female: 1968 (32.6%)
  Male: 4064 (67.4%)

AGE:
  Mean: 38.27
  Std: 13.33
  Range: [17, 90]

AGE_GROUP:
  18-25: 1193 (19.8%)
  26-40: 2408 (39.9%)
  41-60: 204

In [8]:
  # Train the standard (non-private) model. How?
  # Use StandardTrainer class to conduct non-private training
  # Apply training using split training data
  print("\nTraining Model w/o Differential Privacy...")
  print("=" * 60)
  std_model = create_model(input_dim=n_features, hidden_layers=[32, 16])
  std_trainer = StandardTrainer(std_model)
  std_model_trained, std_history = std_trainer.train(X_train, y_train, X_test, y_test)

  print("\nStandard Trained Model:")
  print(f"Final accuracy: {std_history['accuracy'][-1]:.3%}")
  print(f"Final loss: {std_history['loss'][-1]:.3f}")

  # Train the differentially private model. How?
  # Use DPSGDTrainer class to conduct differentially private training
  # Apply training using split training data
  print("\nTrain Model with DP-SGD...")
  print("=" * 60)
  dp_model = create_model(input_dim=n_features, hidden_layers=[32, 16], use_dp=True)
  dp_trainer = DPSGDTrainer(dp_model, epsilon=1.0)
  dp_model_trained, dp_history = dp_trainer.train(X_train, y_train, X_test, y_test)

  print("\nDP Trained Model:")
  print(f"Final accuracy: {dp_history['accuracy'][-1]:.3%}")
  print(f"Final loss: {dp_history['loss'][-1]:.3f}")
  print(f"Privacy budget: ε = {dp_history['epsilon'][-1]:.3f}")

  print("\nTraining Results Assessment...")
  print("=" * 60)
  delta_1 = std_history['accuracy'][-1] - dp_history['accuracy'][-1]
  print(f"\n% Accuracy difference between non-privacy model vs DP-SGD model: {delta_1:.3%}")

  # Train using the PATE mechanism. How?
  # Use PATETrainer class to conduct differentially private training using teacher/student technique
  # Apply training using split training data
  print("\nTraining Model with PATE...")
  print("=" * 60)
  pate_trainer = PATETrainer(input_dim=n_features, hidden_layers=[32, 16])
  pate_history = pate_trainer.train(X_train, y_train, X_test, y_test)

  # For debugging purposes
  #print(type(pate_history))
  #print(pate_history)

  print("\nPATE Trained Model:")
  print(f"Final accuracy: {pate_history[1]['accuracy'][-1]:.3%}")
  print(f"Final loss: {pate_history[1]['loss'][-1]:.3f}")

  print("\nTraining Results Assessment...")
  print("=" * 60)
  delta_2 = std_history['accuracy'][-1] - pate_history[1]['accuracy'][-1]
  print(f"\n% Accuracy difference between non-privacy model vs PATE model: {delta_2:.3%}")


Training Model w/o Differential Privacy...
Epoch 10/50 - Loss: 0.3399 - Val Acc: 0.8408
Epoch 20/50 - Loss: 0.3367 - Val Acc: 0.8404
Epoch 30/50 - Loss: 0.3337 - Val Acc: 0.8400
Epoch 40/50 - Loss: 0.3356 - Val Acc: 0.8425
Epoch 50/50 - Loss: 0.3330 - Val Acc: 0.8413

Standard Trained Model:
Final accuracy: 84.135%
Final loss: 0.333

Train Model with DP-SGD...
Epoch 10/50 - Loss: 0.6598 - Val Acc: 0.7510 - ε: 0.2000
Epoch 20/50 - Loss: 0.5820 - Val Acc: 0.7510 - ε: 0.4000
Epoch 30/50 - Loss: 0.5881 - Val Acc: 0.7510 - ε: 0.6000
Epoch 40/50 - Loss: 0.5936 - Val Acc: 0.7510 - ε: 0.8000
Epoch 50/50 - Loss: 0.5927 - Val Acc: 0.7510 - ε: 1.0000

DP Trained Model:
Final accuracy: 75.099%
Final loss: 0.593
Privacy budget: ε = 1.000

Training Results Assessment...

% Accuracy difference between non-privacy model vs DP-SGD model: 9.035%

Training Model with PATE...
Training 10 teacher models...
  Teacher 1/10 trained
  Teacher 2/10 trained
  Teacher 3/10 trained
  Teacher 4/10 trained
  Teache

In [18]:

  # Perform Label-Only Attack
  # Use LabelOnly class to conduct attack using prediction consistency to determine members vs non-member based on input (x)
  # How? Iterate through samples within the X_train dataset, then using the compute_label_prediction_consistency compute the avg consistency
  # Return predictions and consistency avg
  print("\nPerforming Label-Only Attack on Standard model (For Members)...")
  print("=" * 60)
  lbl_std_obj = LabelOnlyMIA(std_model_trained)
  std_mem_consistency = lbl_std_obj.perform_label_attack_members(X_train, n_samples)
  std_non_member_consistency = lbl_std_obj.perform_label_attack_nonmembers(X_test, n_samples)

  # Define true labels ground truth for analyzing results
  # Creates list containing 1's the len of members_consistency & 0's the len of non_members_consistency (concatenated)
  # Gets the return member & non-member consistency lists above and concatenate them into one list
  y_true_std_consistency = [1] * len(std_mem_consistency) + [0] * len(std_non_member_consistency)
  y_pred_std_consistency = std_mem_consistency + std_non_member_consistency
  auc_std_consistency = roc_auc_score(y_true_std_consistency, y_pred_std_consistency)

  print("\nStandard model Attack Results Assessment...")
  print("=" * 60)
  print(f"Avg Consistency (Members):     {np.mean(std_mem_consistency):.3f}")
  print(f"Avg Consistency (Non-Members): {np.mean(std_non_member_consistency):.3f}")
  print(f"Attack AUC: {auc_std_consistency:.3f}")

  # Assess the Label-Only attacks overall performance
  if auc_std_consistency > 0.5:
    print("nStandard Model Leaks Member Information ~ Label-Only Attack Successful")
  elif auc_std_consistency == 0.5:
    print("Standard Model Has No Privacy Leakage ~ Label-Only Attack Not Successful")


Performing Label-Only Attack on Standard model (For Members)...
Processed: 1000/18097 samples from Members (5.5%)
Processed: 2000/18097 samples from Members (11.1%)
Processed: 3000/18097 samples from Members (16.6%)
Processed: 4000/18097 samples from Members (22.1%)
Processed: 5000/18097 samples from Members (27.6%)
Processed: 6000/18097 samples from Members (33.2%)
Processed: 7000/18097 samples from Members (38.7%)
Processed: 8000/18097 samples from Members (44.2%)
Processed: 9000/18097 samples from Members (49.7%)
Processed: 10000/18097 samples from Members (55.3%)
Processed: 11000/18097 samples from Members (60.8%)
Processed: 12000/18097 samples from Members (66.3%)
Processed: 13000/18097 samples from Members (71.8%)
Processed: 14000/18097 samples from Members (77.4%)
Processed: 15000/18097 samples from Members (82.9%)
Processed: 16000/18097 samples from Members (88.4%)
Processed: 17000/18097 samples from Members (93.9%)
Processed: 18000/18097 samples from Members (99.5%)
Processed

In [10]:
  # Perform Label-Only Attack
  lbl_dp_obj = LabelOnlyMIA(dp_model_trained)

  dp_mem_consistency = lbl_dp_obj.perform_label_attack_members(X_train, n_samples)
  dp_non_member_consistency = lbl_dp_obj.perform_label_attack_members(X_test, n_samples)

  # Define true labels ground truth for analyzing results
  # Creates list containing 1's the len of members_consistency & 0's the len of non_members_consistency (concatenated)
  # Gets the return member & non-member consistency lists above and concatenate them into one list
  y_true_dp_consistency = [1] * len(dp_mem_consistency) + [0] * len(dp_non_member_consistency)
  y_pred_dp_consistency = dp_mem_consistency + dp_non_member_consistency
  auc_dp_consistency = roc_auc_score(y_true_dp_consistency, y_pred_dp_consistency)

  print("\nDP-SGD model Attack Results Assessment...")
  print("=" * 60)
  print(f"Avg Consistency (Members):     {np.mean(dp_mem_consistency):.3f}")
  print(f"Avg Consistency (Non-Members): {np.mean(dp_non_member_consistency):.3f}")
  print(f"Attack AUC: {auc_dp_consistency:.3f}")

  # Assess the Label-Only attacks overall performance
  if auc_dp_consistency > 0.5:
    print("DP-SGD Model Leaks Member Information ~ Label-Only Attack Successful")
  elif auc_dp_consistency == 0.5:
    print("DP-SGD Model Has No Privacy Leakage ~ Label-Only Attack Not Successful")

'# Perform Label-Only Attack\nlbl_dp_obj = LabelOnlyMIA(dp_model_trained)\n\ndp_mem_consistency = lbl_dp_obj.perform_label_attack_members(X_train, n_samples)\ndp_non_member_consistency = lbl_dp_obj.perform_label_attack_members(X_test, n_samples)\n\n# Define true labels ground truth for analyzing results\n# Creates list containing 1\'s the len of members_consistency & 0\'s the len of non_members_consistency (concatenated)\n# Gets the return member & non-member consistency lists above and concatenate them into one list\ny_true_dp_consistency = [1] * len(dp_mem_consistency) + [0] * len(dp_non_member_consistency)\ny_pred_dp_consistency = dp_mem_consistency + dp_non_member_consistency\nauc_dp_consistency = roc_auc_score(y_true_dp_consistency, y_pred_dp_consistency)\n\nprint("\nDP-SGD model Attack Results Assessment...")\nprint("=" * 60)\nprint(f"Avg Consistency (Members):     {np.mean(dp_mem_consistency):.3f}")\nprint(f"Avg Consistency (Non-Members): {np.mean(dp_non_member_consistency):.3f}

In [15]:
  print("\nPerforming Metric-based Loss Attack on Standard Model...")
  print("=" * 60)

  # Setup
  print("\n1. Setup Membership Inference Attack Params...")
  metric_mia = ThresholdBasedMIA(std_trainer, X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], hidden_layers=[32, 16])

  # Train shadow models
  print("\n2. Training Shadow Models...")
  d_train, d_test = metric_mia.train_shadow_model(X_member, y_member, X_nonmember, y_nonmember)

  # Prepare attack dataset
  print("\n3. Preparing dataset for performing attacks...")
  X_attack, y_attack = metric_mia.prep_attack_dataset(d_train, d_test, 'loss')

  #print("X shape:", X_attack.shape)
  #print("y shape:", y_attack.shape)

  # Train attack model
  print("\n4. Train Attack Model...")
  metric_mia.train_attack_model(X_attack, y_attack)

  # Perform the attack
  print("\n5. Performing Attack...")
  results = metric_mia.execute_attack('loss')

  print("\n6. Get Attack Results...")
  print(f"Attack on [members] accuracy: {results['train_accuracy']}")
  print(f"Attack on [non-members] accuracy: {results['test_accuracy']}")
  print(f"Overall accuracy: {results['overall_accuracy']}")

  print("\n7. Threshold Analysis:")

  # Compare accuracy vs baseline of 0.70
  if results['overall_accuracy'] > 0.70:
    print("Standard Model Leaks Member Information ~ Metric Loss Attack Successful")
  else:
    print("Standard Model Leaks Member Information ~ Metric Loss Attack Not Successful")


Performing Metric-based Loss Attack on Standard Model...

1. Setup Membership Inference Attack Params...

2. Training Shadow Models...
Epoch 10/50 - Loss: 0.3460 - Val Acc: 0.6555
Epoch 20/50 - Loss: 0.3357 - Val Acc: 0.6403
Epoch 30/50 - Loss: 0.3315 - Val Acc: 0.6512
Epoch 40/50 - Loss: 0.3258 - Val Acc: 0.6542
Epoch 50/50 - Loss: 0.3284 - Val Acc: 0.6427
Epoch 10/50 - Loss: 0.3456 - Val Acc: 0.6565
Epoch 20/50 - Loss: 0.3420 - Val Acc: 0.6348
Epoch 30/50 - Loss: 0.3327 - Val Acc: 0.6529
Epoch 40/50 - Loss: 0.3348 - Val Acc: 0.6529
Epoch 50/50 - Loss: 0.3282 - Val Acc: 0.6606
Epoch 10/50 - Loss: 0.3501 - Val Acc: 0.6741
Epoch 20/50 - Loss: 0.3378 - Val Acc: 0.6507
Epoch 30/50 - Loss: 0.3398 - Val Acc: 0.6441
Epoch 40/50 - Loss: 0.3329 - Val Acc: 0.6722
Epoch 50/50 - Loss: 0.3350 - Val Acc: 0.6562
Trained 3/3 shadow models

3. Preparing dataset for performing attacks...

4. Train Attack Model...
Epoch 10/50 - Loss: 0.6098 - Val Acc: 0.2104
Epoch 20/50 - Loss: 0.6086 - Val Acc: 0.1915

AttributeError: 'MLP' object has no attribute 'predict_proba'

In [16]:
  print("\nPerforming Confidence-based Loss Attack on Standard Model...")
  print("=" * 60)

  # Setup
  print("\n1. Setup Membership Inference Attack Params...")
  confidence_mia = ThresholdBasedMIA(std_trainer, X_train, y_train, X_test, y_test, input_dim=X_train.shape[1], hidden_layers=[32, 16])

  # Train shadow models
  print("\n2. Training Shadow Models...")
  d_train_2, d_test_2 = confidence_mia.train_shadow_model(X_member, y_member, X_nonmember, y_nonmember)

  # Prepare attack dataset
  print("\n3. Preparing dataset for performing attacks...")
  X_attack_2, y_attack_2 = confidence_mia.prep_attack_dataset(d_train_2, d_test_2, 'confidence')

  #print("X shape:", X_attack.shape)
  #print("y shape:", y_attack.shape)

  # Train attack model
  print("\n4. Train Attack Model...")
  confidence_mia.train_attack_model(X_attack_2, y_attack_2)

  # Perform the attack
  print("\n5. Performing Attack...")
  results_2 = confidence_mia.execute_attack('confidence')

  print("\n6. Get Attack Results...")
  print(f"Attack on [members] accuracy: {results_2['train_accuracy']}")
  print(f"Attack on [non-members] accuracy: {results_2['test_accuracy']}")
  print(f"Overall accuracy: {results_2['overall_accuracy']}")

  print("\n7. Threshold Analysis:")

  # Compare accuracy vs baseline of 0.72
  if results_2['overall_accuracy'] > 0.72:
    print("Standard Model Leaks Member Information ~ Confidence Based Attack Successful")
  else:
    print("Standard Model Leaks Member Information ~ Confidence Based Attack Not Successful")


Performing Confidence-based Loss Attack on Standard Model...

1. Setup Membership Inference Attack Params...

2. Training Shadow Models...
Epoch 10/50 - Loss: 0.3440 - Val Acc: 0.6698
Epoch 20/50 - Loss: 0.3384 - Val Acc: 0.6427
Epoch 30/50 - Loss: 0.3353 - Val Acc: 0.6383
Epoch 40/50 - Loss: 0.3294 - Val Acc: 0.6635
Epoch 50/50 - Loss: 0.3291 - Val Acc: 0.6565
Epoch 10/50 - Loss: 0.3499 - Val Acc: 0.6344
Epoch 20/50 - Loss: 0.3379 - Val Acc: 0.6479
Epoch 30/50 - Loss: 0.3362 - Val Acc: 0.6650
Epoch 40/50 - Loss: 0.3318 - Val Acc: 0.6310
Epoch 50/50 - Loss: 0.3322 - Val Acc: 0.6548
Epoch 10/50 - Loss: 0.3486 - Val Acc: 0.6611
Epoch 20/50 - Loss: 0.3402 - Val Acc: 0.6719
Epoch 30/50 - Loss: 0.3282 - Val Acc: 0.6664
Epoch 40/50 - Loss: 0.3287 - Val Acc: 0.6592
Epoch 50/50 - Loss: 0.3338 - Val Acc: 0.6422
Trained 3/3 shadow models

3. Preparing dataset for performing attacks...


IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (6033,) (6032,) 